In [1]:
FRAMEWORK = 'dask'

# Proyecto Big Data

## 0. Instalación, entorno y acceso a los datos

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q "dask[dataframe]" pyarrow pandas gcsfs
print('Entorno:', 'Google Colab' if IN_COLAB else 'local')

Entorno: local


In [3]:
from pathlib import Path

# El dataset se lee directamente desde el bucket de Google Cloud Storage,
# que es el data lake del proyecto. De esta forma el procesamiento consume
# los datos del bucket y no una copia local.
BUCKET = 'gs://bank-segmentation-bigdata-data'
DATA_PATH = f'{BUCKET}/raw/bank_transactions.csv'

if IN_COLAB:
    # Autenticación necesaria para que Colab pueda leer del bucket.
    from google.colab import auth
    auth.authenticate_user()

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / FRAMEWORK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset: {DATA_PATH}')
print(f'Resultados: {OUTPUT_DIR}')

Dataset: gs://bank-segmentation-bigdata-data/raw/bank_transactions.csv
Resultados: /home/neo/nb2/entrega_polars_dask/outputs/dask


## 1. Carga e inspección con Dask

In [4]:
import math
import pandas as pd
import dask
import dask.dataframe as dd

AMOUNT = 'TransactionAmount (INR)'
BALANCE = 'CustAccountBalance'

# dtypes explícitos
DTYPES = {
    'TransactionID': 'string',
    'CustomerID': 'string',
    'CustomerDOB': 'string',
    'CustGender': 'string',
    'CustLocation': 'string',
    BALANCE: 'float64',
    'TransactionDate': 'string',
    'TransactionTime': 'int64',
    AMOUNT: 'float64',
}

# Definir tamaño de partición
raw = dd.read_csv(
    DATA_PATH, dtype=DTYPES, keep_default_na=False,
    na_values=['', 'nan'], blocksize='32MB',
)

n_raw = int(raw.shape[0].compute())
print('Versión Dask:', dask.__version__)
print('Scheduler:', dask.config.get('scheduler', 'threads (por defecto)'))
print('Particiones:', raw.npartitions)
print('Filas:', n_raw)
display(raw.head())
print(raw.dtypes)

Versión Dask: 2026.8.0
Scheduler: threads (por defecto)
Particiones: 2
Filas: 1048567


,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
0,T1,C5841053,10/1/94,F,JAMSHEDPUR,17819.05,2/8/16,143207,25.0
1,T2,C2142763,4/4/57,M,JHAJJAR,2270.69,2/8/16,141858,27999.0
2,T3,C4417068,26/11/96,F,MUMBAI,17874.44,2/8/16,142712,459.0
3,T4,C5342380,14/9/73,F,MUMBAI,866503.21,2/8/16,142714,2060.0
4,T5,C9031234,24/3/88,F,NAVI MUMBAI,6714.43,2/8/16,181156,1762.5


TransactionID              string[pyarrow]
CustomerID                 string[pyarrow]
CustomerDOB                string[pyarrow]
CustGender                 string[pyarrow]
CustLocation               string[pyarrow]
CustAccountBalance                 float64
TransactionDate            string[pyarrow]
TransactionTime                      int64
TransactionAmount (INR)            float64
dtype: object


## 2. Preparación común236

In [5]:
FRANJAS = {0: 'Madrugada', 1: 'Manana', 2: 'Tarde', 3: 'Noche'}

def preparar(pdf):
    out = pdf.copy()

    transaction_date = pd.to_datetime(
        out['TransactionDate'], format='%d/%m/%y', errors='coerce'
    )
    out['TransactionDateParsed'] = transaction_date
    out['LocationNormalized'] = out['CustLocation'].str.strip().str.upper()

    dob_parts = out['CustomerDOB'].str.split('/')
    dob_day = pd.to_numeric(dob_parts.str[0], errors='coerce')
    dob_month = pd.to_numeric(dob_parts.str[1], errors='coerce')
    dob_year_raw = pd.to_numeric(dob_parts.str[2], errors='coerce')

    txn_year = transaction_date.dt.year
    short_year = (dob_year_raw < 100).fillna(False)
    current_century = (
        (dob_year_raw < 100) & (dob_year_raw <= (txn_year % 100))
    ).fillna(False)
    birth_year = dob_year_raw.where(
        ~short_year, dob_year_raw + 1900 + 100 * current_century.astype('int64')
    )

    birthday_not_reached = (
        (transaction_date.dt.month < dob_month)
        | (
            (transaction_date.dt.month == dob_month)
            & (transaction_date.dt.day < dob_day)
        )
    ).fillna(False).astype('int64')

    age = txn_year - birth_year - birthday_not_reached
    out['Age'] = age.where(age.between(18, 100))

    out['Hour'] = out['TransactionTime'] // 10000
    out['TimeBand'] = (out['Hour'] // 6).map(FRANJAS).fillna('Invalida').astype('object')
    out['AgeRange'] = pd.cut(
        out['Age'],
        bins=[17, 25, 35, 45, 60, 100],
        labels=['18-25', '26-35', '36-45', '46-60', '61-100'],
    ).astype('object').fillna('Desconocido')

    return out

meta_base = preparar(raw._meta_nonempty).iloc[:0]
base = raw.map_partitions(preparar, meta=meta_base)

clean = base.drop_duplicates(subset=['TransactionID']).drop_duplicates(
    subset=['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]
)

clean = clean.persist()
n_clean = int(clean.shape[0].compute())
print('Filas preparadas:', n_clean)
print('Particiones tras limpiar:', clean.npartitions)
print('Filas por partición:', clean.map_partitions(len).compute().tolist())
display(clean.head())

Filas preparadas: 1048567
Particiones tras limpiar: 2
Filas por partición: [524110, 524457]


,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR),TransactionDateParsed,LocationNormalized,Age,Hour,TimeBand,AgeRange
7,T8,C1220223,27/1/82,M,MUMBAI,95075.54,2/8/16,170537,148.00,2016-08-02,MUMBAI,34.0,17,Tarde,26-35
16,T17,C1376215,1/1/1800,M,MUMBAI,77495.15,1/8/16,124727,1423.11,2016-08-01,MUMBAI,NaN,12,Tarde,Desconocido
18,T19,C3732016,11/1/91,M,MUMBAI,32816.17,1/8/16,122135,315.00,2016-08-01,MUMBAI,25.0,12,Tarde,18-25
19,T20,C8999019,24/6/85,M,PUNE,10643.50,1/8/16,152821,945.00,2016-08-01,PUNE,31.0,15,Tarde,26-35
20,T21,C6121429,20/4/93,M,NO 3 KALYANI NAGAR PUNE,2934.22,1/8/16,152824,36.00,2016-08-01,NO 3 KALYANI NAGAR PUNE,23.0,15,Tarde,18-25


## Pregunta 1. Diagnóstico de calidad de datos

In [6]:
%%time
null_counts = raw.isna().sum().compute()
q01 = pd.DataFrame({
    'column': list(raw.columns),
    'null_count': [int(null_counts[c]) for c in raw.columns],
})
q01['null_percent'] = (q01['null_count'] * 100 / n_raw).round(4)
q01.to_csv(OUTPUT_DIR / 'q01_calidad_datos.csv', index=False)
display(q01)

,column,null_count,null_percent
0,TransactionID,0,0.0000
1,CustomerID,0,0.0000
2,CustomerDOB,3397,0.3240
3,CustGender,1100,0.1049
4,CustLocation,151,0.0144
5,CustAccountBalance,2369,0.2259
6,TransactionDate,0,0.0000
7,TransactionTime,0,0.0000
8,TransactionAmount (INR),0,0.0000


CPU times: user 3.62 s, sys: 377 ms, total: 4 s
Wall time: 3.43 s


## Pregunta 2. Detección y tratamiento de duplicados

In [7]:
%%time
def duplicate_excess(columns):
    counts = raw.groupby(columns, dropna=False).size().compute()
    return int((counts[counts > 1] - 1).sum())

q02 = pd.DataFrame({
    'criterion': [
        'TransactionID',
        'CustomerID+TransactionDate+Amount',
        'CustomerID+TransactionDate+Time+Amount',
    ],
    'duplicate_rows': [
        duplicate_excess(['TransactionID']),
        duplicate_excess(['CustomerID', 'TransactionDate', AMOUNT]),
        duplicate_excess(['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]),
    ],
    'treatment': [
        'Eliminar repetidos',
        'Conservar y revisar: ocurren a horas distintas',
        'Eliminar repetidos exactos del evento',
    ],
})
q02.to_csv(OUTPUT_DIR / 'q02_duplicados.csv', index=False)
display(q02)

,criterion,duplicate_rows,treatment
0,TransactionID,0,Eliminar repetidos
1,CustomerID+TransactionDate+Amount,31,Conservar y revisar: ocurren a horas distintas
2,CustomerID+TransactionDate+Time+Amount,0,Eliminar repetidos exactos del evento


CPU times: user 16.1 s, sys: 801 ms, total: 16.9 s
Wall time: 13.8 s


## Pregunta 3. Edad exacta y rangos etarios

In [8]:
%%time
q03 = clean.groupby('AgeRange').agg(
    transactions=('TransactionID', 'size'),
    mean_age=('Age', 'mean'),
).compute().reset_index()
q03['mean_age'] = q03['mean_age'].round(2)
q03 = q03.sort_values('AgeRange').reset_index(drop=True)
q03.to_csv(OUTPUT_DIR / 'q03_edades.csv', index=False)
display(q03)

,AgeRange,transactions,mean_age
0,18-25,295117,23.08
1,26-35,482990,29.62
2,36-45,142585,39.43
3,46-60,50856,51.03
4,61-100,14303,67.10
5,Desconocido,62716,NaN


CPU times: user 179 ms, sys: 4.23 ms, total: 183 ms
Wall time: 150 ms


## Pregunta 4. Franja horaria y ubicación normalizada

In [9]:
%%time
transactions = clean.groupby('TimeBand')['TransactionID'].size().compute()
unique_locations = clean.groupby('TimeBand')['LocationNormalized'].nunique().compute()

q04 = pd.concat(
    [transactions.rename('transactions'), unique_locations.rename('unique_locations')],
    axis=1,
).reset_index(names='TimeBand').sort_values(
    ['transactions', 'TimeBand'], ascending=[False, True]
).reset_index(drop=True)
q04.to_csv(OUTPUT_DIR / 'q04_franja_ubicacion.csv', index=False)
display(q04)

,TimeBand,transactions,unique_locations
0,Noche,436179,7246
1,Tarde,390884,7146
2,Manana,174473,4889
3,Madrugada,47031,2208


CPU times: user 419 ms, sys: 30.7 ms, total: 450 ms
Wall time: 420 ms


## Pregunta 5. Outliers por percentiles 1 y 99

In [10]:
%%time
balance_p01, balance_p99 = clean[BALANCE].quantile([0.01, 0.99]).compute().tolist()
amount_p01, amount_p99 = clean[AMOUNT].quantile([0.01, 0.99]).compute().tolist()

balance_outliers = int(
    ((clean[BALANCE] < balance_p01) | (clean[BALANCE] > balance_p99)).sum().compute()
)
amount_outliers = int(
    ((clean[AMOUNT] < amount_p01) | (clean[AMOUNT] > amount_p99)).sum().compute()
)

q05 = pd.DataFrame({
    'variable': [BALANCE, AMOUNT],
    'p01': [balance_p01, amount_p01],
    'p99': [balance_p99, amount_p99],
    'outlier_rows': [balance_outliers, amount_outliers],
})
q05.to_csv(OUTPUT_DIR / 'q05_outliers.csv', index=False)
display(q05)

,variable,p01,p99,outlier_rows
0,CustAccountBalance,3.28,1587419.41,20933
1,TransactionAmount (INR),8.00,20000.00,20407


CPU times: user 1.15 s, sys: 3.63 ms, total: 1.15 s
Wall time: 1.1 s


## Pregunta 6. Balance y monto promedio por género y edad

In [11]:
%%time
valid_demo = clean[clean['CustGender'].notnull() & clean['Age'].notnull()]
q06 = valid_demo.groupby(['CustGender', 'AgeRange']).agg(
    transactions=('TransactionID', 'size'),
    avg_balance=(BALANCE, 'mean'),
    avg_amount=(AMOUNT, 'mean'),
).compute().reset_index()
q06[['avg_balance', 'avg_amount']] = q06[['avg_balance', 'avg_amount']].round(2)
q06 = q06.sort_values(['CustGender', 'AgeRange']).reset_index(drop=True)
q06.to_csv(OUTPUT_DIR / 'q06_genero_edad.csv', index=False)
display(q06)

,CustGender,AgeRange,transactions,avg_balance,avg_amount
0,F,18-25,94727,37858.98,1007.18
1,F,26-35,125778,84128.28,1620.86
2,F,36-45,34248,200026.15,2323.98
3,F,46-60,14126,267580.73,3171.41
4,F,61-100,4210,695495.59,3080.99
5,M,18-25,200390,33768.21,803.19
6,M,26-35,357212,84804.58,1275.34
7,M,36-45,108337,190350.65,2155.85
8,M,46-60,36730,348567.61,2973.25
9,M,61-100,9942,649675.48,3703.29


CPU times: user 1.34 s, sys: 60.5 ms, total: 1.4 s
Wall time: 1.25 s


## Pregunta 7. Top 20 ciudades

In [12]:
%%time
q07 = clean[clean['LocationNormalized'].notnull()].groupby('LocationNormalized').agg(
    transactions=('TransactionID', 'size'),
    total_amount=(AMOUNT, 'sum'),
).compute().reset_index()
q07['total_amount'] = q07['total_amount'].round(2)
q07 = q07.sort_values(
    ['total_amount', 'LocationNormalized'], ascending=[False, True]
).head(20).reset_index(drop=True)
q07.to_csv(OUTPUT_DIR / 'q07_top_ciudades.csv', index=False)
display(q07)

,LocationNormalized,transactions,total_amount
0,MUMBAI,103596,1.796891e+08
1,NEW DELHI,84928,1.607059e+08
2,BANGALORE,81555,1.184248e+08
3,GURGAON,73818,1.120947e+08
4,DELHI,71019,1.062249e+08
5,KOLKATA,19974,6.060031e+07
6,CHENNAI,30009,4.463782e+07
7,NOIDA,32784,4.446343e+07
8,PUNE,25851,3.959035e+07
9,HYDERABAD,23049,3.617739e+07


CPU times: user 739 ms, sys: 19.9 ms, total: 759 ms
Wall time: 674 ms


## Pregunta 8. Cliente con mayor gasto por ciudad

In [13]:
%%time
spending = clean[
    clean['LocationNormalized'].notnull() & clean['CustomerID'].notnull()
].groupby(['LocationNormalized', 'CustomerID']).agg(
    total_spent=(AMOUNT, 'sum'),
    transactions=('TransactionID', 'size'),
).reset_index()

def top1_por_ciudad(pdf):
    return (
        pdf.reset_index()
        .sort_values(
            ['LocationNormalized', 'total_spent', 'CustomerID'],
            ascending=[True, False, True],
        )
        .groupby('LocationNormalized', as_index=False)
        .head(1)
    )

spending_idx = spending.set_index('LocationNormalized')
meta_top = top1_por_ciudad(spending_idx._meta_nonempty).iloc[:0]

q08 = spending_idx.map_partitions(top1_por_ciudad, meta=meta_top).compute()
q08['city_rank'] = 1
q08['total_spent'] = q08['total_spent'].round(2)
q08 = q08[[
    'LocationNormalized', 'CustomerID', 'total_spent', 'transactions', 'city_rank'
]].sort_values(
    ['total_spent', 'LocationNormalized'], ascending=[False, True]
).reset_index(drop=True)

q08.to_csv(OUTPUT_DIR / 'q08_top_cliente_ciudad.csv', index=False)
print('Ciudades:', len(q08))
display(q08.head(20))

Ciudades: 9353


,LocationNormalized,CustomerID,total_spent,transactions,city_rank
0,GURGAON,C7319271,1560034.99,1,1
1,PUNE,C6677159,1380002.88,1,1
2,NEW DELHI,C4141768,991132.22,1,1
3,MUMBAI,C8217728,724122.00,1,1
4,KOLKATA,C1830891,720001.16,1,1
5,NOIDA,C6549785,600008.32,1,1
6,DELHI,C4328064,569500.27,1,1
7,PALAKKARAI TRICHY,C5833636,557000.73,1,1
8,LUDHIANA,C8755262,514320.00,1,1
9,BANGALORE,C5720892,500000.00,1,1


CPU times: user 6.76 s, sys: 307 ms, total: 7.07 s
Wall time: 6.82 s


## Pregunta 9. Serie temporal diaria

In [14]:
%%time
q09 = clean[clean['TransactionDateParsed'].notnull()].groupby(
    'TransactionDateParsed'
).agg(
    transactions=('TransactionID', 'size'),
    total_amount=(AMOUNT, 'sum'),
).compute().reset_index()
q09['total_amount'] = q09['total_amount'].round(2)
q09 = q09.sort_values('TransactionDateParsed').reset_index(drop=True)
q09.to_csv(OUTPUT_DIR / 'q09_serie_diaria.csv', index=False)
display(q09)

,TransactionDateParsed,transactions,total_amount
0,2016-08-01,20438,29801816.34
1,2016-08-02,20948,30467503.29
2,2016-08-03,20615,31149483.67
3,2016-08-04,20682,35722718.64
4,2016-08-05,21112,34833933.12
5,2016-08-06,26585,47527227.82
6,2016-08-07,27261,45727772.63
7,2016-08-08,21042,30129433.64
8,2016-08-09,21823,33479570.48
9,2016-08-10,21649,32016308.51


CPU times: user 597 ms, sys: 153 μs, total: 597 ms
Wall time: 571 ms


## Pregunta 10. Ratio gasto/balance y top 1%

In [15]:
%%time
valid = clean[
    clean[BALANCE].notnull() & (clean[BALANCE] > 0) & clean[AMOUNT].notnull()
]
valid = valid.assign(spend_balance_ratio=valid[AMOUNT] / valid[BALANCE])

n_valid = int(valid.shape[0].compute())
top_count = math.ceil(n_valid * 0.01)

pool = valid.nlargest(top_count * 2, 'spend_balance_ratio').compute()
q10 = pool.sort_values(
    ['spend_balance_ratio', 'TransactionID'], ascending=[False, True]
).head(top_count)
q10 = q10[[
    'TransactionID', 'CustomerID', 'LocationNormalized',
    BALANCE, AMOUNT, 'spend_balance_ratio',
]].reset_index(drop=True)

q10.to_csv(OUTPUT_DIR / 'q10_ratio_top1.csv', index=False)
print('Filas válidas:', n_valid, '| Filas del top 1%:', len(q10))
display(q10.head(20))

Filas válidas: 1043487 | Filas del top 1%: 10435


,TransactionID,CustomerID,LocationNormalized,CustAccountBalance,TransactionAmount (INR),spend_balance_ratio
0,T742111,C7323566,CHANDIGARH,0.01,42398.0,4239800.0
1,T836117,C5719489,KOLAR,0.01,25500.0,2550000.0
2,T253453,C3523520,CHANDIGARH,0.01,20000.0,2000000.0
3,T652735,C6038911,TANK HYDERABAD,0.01,17820.0,1782000.0
4,T421850,C2816416,MUMBAI,0.01,15715.0,1571500.0
5,T424045,C7338987,TANK HYDERABAD,0.01,10764.0,1076400.0
6,T343016,C2223587,CHANDIGARH,0.01,9500.0,950000.0
7,T782382,C8238539,LUCKNOW,0.01,9000.0,900000.0
8,T311365,C5527984,FARIDABAD,0.01,8700.0,870000.0
9,T196759,C1920771,CHENNAI,0.05,42625.0,852500.0


CPU times: user 3.15 s, sys: 163 ms, total: 3.31 s
Wall time: 3.01 s
